# Gene Set Variation Analysis



https://davetang.github.io/muse/gsva.html

https://www.biostars.org/p/407614/

https://medium.com/data-science/decoding-gene-set-variation-analysis-8193a0cfda3

https://pmc.ncbi.nlm.nih.gov/articles/PMC9882404/

In [1]:
import os

repo_url = "https://github.com/chenzRG/Cancer-Multi-Omics-Benchmark.git"
repo_dir = "Cancer-Multi-Omics-Benchmark"

# Move to root if in src folder
if os.path.basename(os.getcwd()) == 'src':
    os.chdir("..")
    print("Moved to root directory")

# Clone or pull
if os.path.exists(repo_dir):
    print(f"Pulling latest changes...")
    !cd {repo_dir} && git pull
else:
    print(f"Cloning repository...")
    !git clone {repo_url}

Moved to root directory
Pulling latest changes...
Already up to date.


In [2]:
import pandas as pd
from huggingface_hub import hf_hub_download
import matplotlib.pyplot as plt

plt.style.use('dark_background')

repo_id = "AIBIC/MLOmics"
base = "Main_Dataset/Classification_datasets/GS-BRCA"

versions = {
    "Original": {
        "CNV":   "BRCA_CNV.csv",
        "Methy": "BRCA_Methy.csv",
        "mRNA":  "BRCA_mRNA.csv",
        "miRNA": "BRCA_miRNA.csv",
        "label": "BRCA_label_num.csv",
    },
    "Top": {
        "CNV":   "BRCA_CNV_top.csv",
        "Methy": "BRCA_Methy_top.csv",
        "mRNA":  "BRCA_mRNA_top.csv",
        "miRNA": "BRCA_miRNA_top.csv",
        "label": "BRCA_label_num.csv",
    },
    "Aligned": {
        "CNV":   "BRCA_CNV_aligned.csv",
        "Methy": "BRCA_Methy_aligned.csv",
        "mRNA":  "BRCA_mRNA_aligned.csv",
        "miRNA": "BRCA_miRNA_aligned.csv",
        "label": "BRCA_label_num.csv",
    },
}

all_dfs = {}  # all_dfs["Original"]["CNV"], etc.

for version, files in versions.items():
    print(f"\n{'='*40}")
    print(f"  {version}")
    print(f"{'='*40}")
    all_dfs[version] = {}

    for name, filename in files.items():

        if os.path.exists(f"data/{base}/{version}/{filename}"):
            path = f"data/{base}/{version}/{filename}"
        else:
            path = hf_hub_download(
                repo_id=repo_id,
                filename=f"{base}/{version}/{filename}",
                repo_type="dataset")
        
        df = pd.read_csv(path, index_col=0)

        if name == "label":
            df = df.index.to_series().reset_index(drop=True)
            all_dfs[version][name] = df
            print(f"  {name}: {df.shape} → value counts: {df.value_counts().to_dict()}")
        else:
            all_dfs[version][name] = df
            print(f"  {name}: {df.shape}")


  Original


  CNV: (19568, 671)
  Methy: (19049, 671)
  mRNA: (18206, 671)
  miRNA: (368, 671)
  label: (671,) → value counts: {0: 353, 2: 132, 4: 113, 1: 42, 3: 31}

  Top
  CNV: (5000, 671)
  Methy: (5000, 671)
  mRNA: (5000, 671)
  miRNA: (366, 671)
  label: (671,) → value counts: {0: 353, 2: 132, 4: 113, 1: 42, 3: 31}

  Aligned
  CNV: (11203, 671)
  Methy: (11189, 671)
  mRNA: (11343, 671)
  miRNA: (310, 671)
  label: (671,) → value counts: {0: 353, 2: 132, 4: 113, 1: 42, 3: 31}


In [ ]:
import gseapy

mrna = all_dfs['Original']['mRNA']  # genes x patients

res_gsva = gseapy.gsva(
    data=mrna,  # genes x patients
    gene_sets='KEGG_2021_Human',
    method='gsva',
    kcdf='Gaussian',  #continuous expression data
    outdir=None,
    no_plot=True,
)

gsva_scores = res_gsva.res2d.pivot(index='Name', columns='Term', values='ES')
print(f"GSVA scores: {gsva_scores.shape}")  # 671 x ~298

GSVA scores: (671, 298)


In [ ]:
gsva_scores.iloc[:5,:5]

Term,ABC transporters,AGE-RAGE signaling pathway in diabetic complications,AMPK signaling pathway,Acute myeloid leukemia,Adherens junction,Adipocytokine signaling pathway,Adrenergic signaling in cardiomyocytes,African trypanosomiasis,"Alanine, aspartate and glutamate metabolism",Alcoholism,...,Vitamin digestion and absorption,Wnt signaling pathway,Yersinia infection,alpha-Linolenic acid metabolism,beta-Alanine metabolism,cAMP signaling pathway,cGMP-PKG signaling pathway,mRNA surveillance pathway,mTOR signaling pathway,p53 signaling pathway
Name,,,,,,,,,,,,,,,,,,,,,
TCGA.3C.AAAU.01,-0.114022,-0.131259,-0.007165,-0.088124,-0.121120,-0.179089,-0.018681,-0.592806,-0.073980,0.042030,...,-0.177626,-0.116189,-0.137838,0.070189,-0.276253,-0.074745,-0.104212,0.038052,0.087933,-0.123051
TCGA.3C.AALI.01,0.107260,-0.128425,0.027969,0.055378,-0.034198,-0.067948,0.035222,0.420954,0.066894,-0.005144,...,-0.273653,-0.104653,0.013134,-0.112837,-0.023928,-0.028225,-0.043032,-0.132533,0.045830,0.108730
TCGA.3C.AALJ.01,-0.197496,-0.112714,0.017153,-0.151270,-0.102246,-0.036335,0.008203,0.058756,-0.157912,0.016581,...,-0.049139,-0.194753,-0.053243,-0.119646,-0.315592,-0.108197,-0.035375,0.163771,-0.070882,0.204292
TCGA.3C.AALK.01,0.146460,0.360674,-0.062391,-0.064430,0.305997,0.097039,0.176858,0.062847,-0.240959,0.038972,...,0.024366,0.288948,0.088915,0.026721,-0.184279,0.303057,0.227318,-0.186636,0.050072,0.199936
TCGA.5L.AAT0.01,-0.169027,-0.018869,-0.318143,-0.215698,-0.162184,-0.180182,-0.060161,0.109619,-0.322420,-0.067085,...,-0.114017,0.140494,-0.138428,0.020959,-0.062146,-0.158807,-0.169202,-0.074712,-0.202367,-0.291881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA.WT.AB44.01,-0.005117,-0.105289,-0.156405,-0.345190,-0.155712,-0.089022,-0.012214,0.053254,0.026197,-0.097379,...,0.082883,-0.038910,-0.111745,0.054031,-0.192068,0.103475,-0.122463,-0.216763,-0.195961,-0.251845
TCGA.XX.A899.01,0.335875,0.342339,0.188126,0.235353,0.219194,0.356785,0.135520,0.314922,0.086433,-0.046648,...,0.130531,0.051372,0.241699,-0.208345,0.103667,0.274780,0.232502,-0.312799,0.073346,0.025819
TCGA.XX.A89A.01,0.341159,0.269712,0.241610,0.276158,0.081597,0.430856,0.155949,0.063705,0.177052,0.064267,...,0.234426,0.200597,0.143984,0.482949,0.440545,0.226331,0.205002,-0.317824,-0.095825,0.072321


In [9]:
gsva_dict = {}
for name in ['CNV', 'Methy', 'mRNA']:
    df = all_dfs['Original'][name]
    print(f"{name}: {df.shape} → ", end="")
    res = gseapy.gsva(
        data=df,
        gene_sets='KEGG_2021_Human',
        method='gsva',
        kcdf='Gaussian',
        outdir=None,
        no_plot=True,
    )
    gsva_dict[name] = res.res2d.pivot(index='Name', columns='Term', values='ES')
    print(gsva_dict[name].shape)

CNV: (19568, 671) → (671, 305)
Methy: (19049, 671) → (671, 305)
mRNA: (18206, 671) → (671, 298)
